In [1]:
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import SGDClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler


In [3]:
import copy

In [4]:
data = load_breast_cancer()

In [24]:
x_train, x_test, Y_train, Y_test = train_test_split(data.data, data.target, test_size=0.25)

In [25]:
clf = SGDClassifier(loss='log', max_iter=10000, alpha=0.001)
clf.fit(x_train, Y_train)
print('roc_auc: ',roc_auc_score(Y_test, clf.predict_proba(x_test)[:,1]))
print('accuracy: ', accuracy_score(Y_test, clf.predict(x_test)))

roc_auc:  0.880498866213152
accuracy:  0.9020979020979021


In [ ]:
class PU_model:
    def __init__(self, model=SGDClassifier(loss='log', max_iter=10000, alpha=0.001),
                 estimator_type=1, weighted=False):
        """ model_s - модель g(x), оценивающая вероятность p(s=1|x),
        estimator_type принимает значения 1,2,3, сооствествующие способам расчета константы (self.const),
        weighted: False - алгоритм из второй главы(возвращает констату с), 
        True - из третьей, в котором множеству без меток (U) присваивается значение y=1,
        создается копия множества U, которому присваивается y=0, а затем алгоритм обучается заново 
        на объектах из U с весом w при y=1 и 1-w при y=0ю Возвращает None"""
        self.model= model
        self.estimator_type = estimator_type
        self.const = []
        self.cv_score = []
        self.weighted = weighted

    def choose_random_s(self, x, y):
        chosen = np.random.randint(0,2, size = y.shape)
        labels = y[(y==1) & (chosen==1)]
        x_labeled = x[(y==1)&(chosen==1)]
        x_unlabeled = x[(y==0)|(chosen==0)]
        y_no_label = y[(y==0)|(chosen==0)]
        no_label = np.zeros(y_no_label.shape[0])
        x=np.concatenate((x_unlabeled, x_labeled), axis = 0)
        y = np.concatenate((y_no_label, labels), axis=0)[:, np.newaxis]
        s = np.concatenate((no_label, labels), axis=0)[:, np.newaxis]
        ys = np.concatenate((y, s), axis=1)
        return x, ys

    def train(self, X_train, ys_train, X_val, ys_val):
        self.model.fit(X_train, ys_train[:,1])
        kf = KFold(n_splits=3)
        for train, test in kf.split(X_val):
            g =self.model.predict_proba(X_val)[train,1]
            c = self.estimator(g, ys_val[train,1])
            self.const.append(c)
            self.cv_score.append(self.test(X_val[test], ys_val[test, 0], c)[0])
        const = self.const[self.cv_score.index(max(self.cv_score))]
        if self.weighted:
            g =self.model.predict_proba(X_val)[:,1]
            y = np.concatenate((np.zeros(ys_val.shape[0]), np.ones(ys_val.shape[0])))
            weights = self.weight(g, const, ys_val[:,1])
            y[weights==1]=1
            self.model.fit(np.concatenate((X_val, X_val), axis = 0), y, sample_weight=weights)
        else:
            return const

    def test(self, X_test, y_test, const):
        proba = self.model.predict_proba(X_test)[:,1]/const
        k=[]
        k.append(roc_auc_score(y_test, proba))
        k.append(accuracy_score(y_test, np.around(proba)))
        return k

    def estimator(self, proba, s):
        if self.estimator_type==1:
            c = np.sum(proba*s)/s[s==1].shape[0]
            return c
        elif self.estimator_type==2:
            return np.sum(proba*s)/np.sum(proba)
        else:
            return np.max(proba)

    def weight(self, g, constant, s):
        w1 = ((1-constant)/constant)*(g/(np.ones(g.shape[0])-g))
        w1 = np.nan_to_num(w1)
        w1[s==1] = 1
        w2 = np.ones(w1.shape[0]) - w1
        return np.concatenate((w1, w2))     

In [42]:
pu = PU_model()
X, ys = pu.choose_random_s(x=data.data, y=data.target)

In [43]:
X_train, X_tv, ys_train, ys_tv  = train_test_split(X, ys, test_size = 0.5)
X_val, X_test, ys_val, ys_test = train_test_split(X_tv, ys_tv, test_size = 0.5)

In [44]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [47]:
c = pu.train(X_train, ys_train, X_val, ys_val)
lst = pu.test(X_test, ys_test[:,0], c)
print('roc_auc_score: ',lst[0])
print('accuracy: ',lst[1])

roc_auc_score:  0.9988521579430671
accuracy:  0.8951048951048951


In [ ]:
pu_w = PU_model(weighted=True)

In [22]:
y = copy.copy(ys_test[:,0])
y[y==1]=0
y[ys_test[:,0]==0]=1

In [23]:
pu_w.train(X_train, ys_train, X_val, ys_val)
lst = pu_w.test(X_test, y, 1)
print('roc_auc_score: ',lst[0])
print('accuracy: ',lst[1])

roc_auc_score:  0.9114255765199162
accuracy:  0.8741258741258742
